# DepegScope: Simulation Scenarios

This notebook runs depeg contagion simulations using the agent-based model.

## Contents
1. Setup and Configuration
2. Single Scenario Simulation
3. Predefined Scenarios
4. Monte Carlo Analysis
5. Sensitivity Analysis
6. Scenario Comparison

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.simulation.environment import DeFiEnvironment
from src.simulation.scenarios import (
    DepegScenario, ScenarioRunner,
    PREDEFINED_SCENARIOS, get_scenario, list_scenarios
)
from config.settings import PROCESSED_DATA_DIR

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

In [ ]:
# Load data for simulation
def load_simulation_data():
    stablecoins_file = PROCESSED_DATA_DIR / "stablecoins_processed.json"
    protocols_file = PROCESSED_DATA_DIR / "protocols_processed.json"
    
    with open(stablecoins_file) as f:
        stablecoins_data = json.load(f)
    
    stablecoins = []
    for s in stablecoins_data:
        stablecoins.append({
            "symbol": s.get("symbol", ""),
            "market_cap": s.get("market_cap", 1e9),
            "collateral": s.get("collateral", []),
            "is_algorithmic": s.get("stablecoin_type") == "algorithmic",
            "stablecoin_type": s.get("stablecoin_type", "unknown")
        })
    
    protocols = []
    if protocols_file.exists():
        with open(protocols_file) as f:
            protocols_data = json.load(f)
        
        for p in protocols_data:
            protocols.append({
                "name": p.get("slug", p.get("name", "")),
                "tvl": p.get("tvl", 1e8),
                "exposures": p.get("stablecoin_holdings", {}),
                "category": p.get("category", "other")
            })
    
    return stablecoins, protocols

stablecoins, protocols = load_simulation_data()
print(f"Loaded: {len(stablecoins)} stablecoins, {len(protocols)} protocols")

## 1. Simulation Configuration

In [ ]:
# Default simulation configuration
DEFAULT_CONFIG = {
    "depeg_threshold": 0.02,           # 2% from peg triggers depeg status
    "max_steps": 100,                   # Maximum simulation steps
    "confidence_decay_rate": 0.1,       # Confidence decay per step under pressure
    "noise_std": 0.01,                  # Price noise standard deviation
    "algorithmic_death_spiral_factor": 0.9,  # Extra severity for algo coins
}

print("Default Configuration:")
for key, value in DEFAULT_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# List available predefined scenarios
print("Available Predefined Scenarios:")
for name in list_scenarios():
    scenario = get_scenario(name)
    print(f"  - {name}: {scenario.description}")
    print(f"    Trigger: {scenario.trigger_stablecoin}, Severity: {scenario.initial_severity*100:.0f}%")

## 2. Single Scenario Simulation

In [ ]:
# Run a single simulation
def run_simulation(trigger, severity, config=None, seed=42):
    if config is None:
        config = DEFAULT_CONFIG.copy()
    
    env = DeFiEnvironment(
        stablecoins=stablecoins,
        protocols=protocols,
        config=config,
        seed=seed
    )
    
    env.trigger_depeg(trigger, severity)
    results = env.run_simulation()
    
    return results, env

In [ ]:
# Run example: USDC 10% depeg
results, env = run_simulation("USDC", 0.10)

print("=== Simulation Results ===")
print(f"Steps to equilibrium: {results['steps']}")
print(f"Total TVL Lost: ${results['total_tvl_lost']:,.0f}")
print(f"Depegged Stablecoins: {results['depegged_stablecoins']}")
print(f"Distressed Protocols: {results['distressed_protocols']}")
print(f"Contagion Index: {results['contagion_index']:.3f}")

In [ ]:
# Visualize simulation timeline
if 'price_history' in results and results['price_history']:
    price_df = pd.DataFrame(results['price_history'])
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    
    # Price evolution
    for col in price_df.columns[:5]:  # Top 5 stablecoins
        axes[0].plot(price_df.index, price_df[col], label=col)
    axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    axes[0].axhline(y=0.98, color='red', linestyle='--', alpha=0.5, label='Depeg threshold')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Price')
    axes[0].set_title('Stablecoin Price Evolution')
    axes[0].legend(loc='lower right')
    
    # TVL evolution
    if 'tvl_history' in results and results['tvl_history']:
        tvl_df = pd.DataFrame(results['tvl_history'])
        total_tvl = tvl_df.sum(axis=1)
        axes[1].plot(total_tvl.index, total_tvl / 1e9)
        axes[1].set_xlabel('Step')
        axes[1].set_ylabel('Total TVL (Billions USD)')
        axes[1].set_title('Total Protocol TVL Evolution')
    
    plt.tight_layout()
    plt.show()
else:
    print("No price history available for visualization")

## 3. Predefined Scenarios

In [ ]:
# Run predefined scenarios
scenario_results = {}

for name in list_scenarios():
    scenario = get_scenario(name)
    results, _ = run_simulation(
        scenario.trigger_stablecoin,
        scenario.initial_severity
    )
    scenario_results[name] = results
    print(f"{name}: TVL Lost = ${results['total_tvl_lost']:,.0f}, Depegged = {results['depegged_stablecoins']}")

In [ ]:
# Compare scenarios
comparison_data = []
for name, results in scenario_results.items():
    comparison_data.append({
        'scenario': name,
        'tvl_lost': results['total_tvl_lost'],
        'steps': results['steps'],
        'depegged': results['depegged_stablecoins'],
        'distressed': results['distressed_protocols'],
        'contagion_index': results['contagion_index']
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison_df.plot(kind='bar', x='scenario', y='tvl_lost', ax=axes[0], legend=False)
axes[0].set_ylabel('TVL Lost (USD)')
axes[0].set_title('TVL Lost by Scenario')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e9:.1f}B'))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

comparison_df.plot(kind='bar', x='scenario', y='contagion_index', ax=axes[1], legend=False, color='orange')
axes[1].set_ylabel('Contagion Index')
axes[1].set_title('Contagion Index by Scenario')
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 4. Monte Carlo Analysis

In [ ]:
# Monte Carlo simulation
def run_monte_carlo(trigger, severity, n_runs=100, config=None):
    if config is None:
        config = DEFAULT_CONFIG.copy()
    
    results_list = []
    
    for i in tqdm(range(n_runs), desc=f"Monte Carlo ({trigger}, {severity*100:.0f}%)"):
        results, _ = run_simulation(trigger, severity, config, seed=i)
        results_list.append({
            'run': i,
            'tvl_lost': results['total_tvl_lost'],
            'steps': results['steps'],
            'depegged': results['depegged_stablecoins'],
            'distressed': results['distressed_protocols'],
            'contagion_index': results['contagion_index']
        })
    
    return pd.DataFrame(results_list)

In [ ]:
# Run Monte Carlo for USDC
mc_results = run_monte_carlo("USDC", 0.10, n_runs=100)

print("=== Monte Carlo Results (USDC 10% Depeg) ===")
print(f"TVL Lost:")
print(f"  Mean: ${mc_results['tvl_lost'].mean():,.0f}")
print(f"  Std Dev: ${mc_results['tvl_lost'].std():,.0f}")
print(f"  95th Percentile: ${mc_results['tvl_lost'].quantile(0.95):,.0f}")
print(f"  99th Percentile: ${mc_results['tvl_lost'].quantile(0.99):,.0f}")
print(f"\nAvg Depegged Stablecoins: {mc_results['depegged'].mean():.1f}")
print(f"Avg Distressed Protocols: {mc_results['distressed'].mean():.1f}")

In [ ]:
# Visualize Monte Carlo distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TVL Loss distribution
axes[0].hist(mc_results['tvl_lost'] / 1e9, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(mc_results['tvl_lost'].mean() / 1e9, color='red', linestyle='--', label='Mean')
axes[0].axvline(mc_results['tvl_lost'].quantile(0.95) / 1e9, color='orange', linestyle='--', label='95th %ile')
axes[0].set_xlabel('TVL Lost (Billions USD)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of TVL Losses')
axes[0].legend()

# Contagion index distribution
axes[1].hist(mc_results['contagion_index'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(mc_results['contagion_index'].mean(), color='red', linestyle='--', label='Mean')
axes[1].set_xlabel('Contagion Index')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Contagion Index')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Sensitivity Analysis

In [ ]:
# Sensitivity to initial severity
severities = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
sensitivity_results = []

for severity in tqdm(severities, desc="Severity sensitivity"):
    results, _ = run_simulation("USDC", severity)
    sensitivity_results.append({
        'severity': severity,
        'tvl_lost': results['total_tvl_lost'],
        'depegged': results['depegged_stablecoins'],
        'contagion_index': results['contagion_index']
    })

sens_df = pd.DataFrame(sensitivity_results)

In [ ]:
# Plot sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(sens_df['severity'] * 100, sens_df['tvl_lost'] / 1e9, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Initial Depeg Severity (%)')
axes[0].set_ylabel('TVL Lost (Billions USD)')
axes[0].set_title('TVL Loss vs Initial Severity')
axes[0].grid(True)

axes[1].plot(sens_df['severity'] * 100, sens_df['depegged'], 'o-', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('Initial Depeg Severity (%)')
axes[1].set_ylabel('Number of Depegged Stablecoins')
axes[1].set_title('Cascade Size vs Initial Severity')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity to confidence decay rate
decay_rates = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
decay_results = []

for decay in tqdm(decay_rates, desc="Decay rate sensitivity"):
    config = DEFAULT_CONFIG.copy()
    config['confidence_decay_rate'] = decay
    results, _ = run_simulation("USDC", 0.10, config)
    decay_results.append({
        'decay_rate': decay,
        'tvl_lost': results['total_tvl_lost'],
        'steps': results['steps']
    })

decay_df = pd.DataFrame(decay_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(decay_df['decay_rate'], decay_df['tvl_lost'] / 1e9, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('Confidence Decay Rate')
ax.set_ylabel('TVL Lost (Billions USD)')
ax.set_title('Impact of Confidence Decay Rate on Losses')
ax.grid(True)
plt.show()

## 6. Multi-Stablecoin Comparison

In [ ]:
# Compare different trigger stablecoins
trigger_stables = ["USDT", "USDC", "DAI", "FRAX", "BUSD"]
available_stables = [s['symbol'] for s in stablecoins]
trigger_stables = [s for s in trigger_stables if s in available_stables]

comparison_results = []

for stable in tqdm(trigger_stables, desc="Comparing stablecoins"):
    results, _ = run_simulation(stable, 0.10)
    comparison_results.append({
        'stablecoin': stable,
        'tvl_lost': results['total_tvl_lost'],
        'depegged': results['depegged_stablecoins'],
        'distressed': results['distressed_protocols'],
        'contagion_index': results['contagion_index']
    })

compare_df = pd.DataFrame(comparison_results)
compare_df = compare_df.sort_values('tvl_lost', ascending=False)
display(compare_df)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(compare_df))
width = 0.35

ax.bar(x - width/2, compare_df['tvl_lost'] / 1e9, width, label='TVL Lost (B)', color='#e74c3c')
ax2 = ax.twinx()
ax2.bar(x + width/2, compare_df['contagion_index'], width, label='Contagion Index', color='#3498db')

ax.set_xlabel('Trigger Stablecoin')
ax.set_ylabel('TVL Lost (Billions USD)', color='#e74c3c')
ax2.set_ylabel('Contagion Index', color='#3498db')
ax.set_title('Impact Comparison: 10% Depeg of Different Stablecoins')
ax.set_xticks(x)
ax.set_xticklabels(compare_df['stablecoin'])

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

## Summary

Key simulation findings:
1. Identified most impactful depeg scenarios
2. Monte Carlo analysis provides loss distributions
3. Sensitivity analysis shows non-linear responses
4. Different stablecoins have varying systemic importance

## Next Steps

Continue to historical validation:
- `04_historical_validation.ipynb`